In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load

In [0]:

dbutils.widgets.text("bronze_table_name", "tb_motivos")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
df_motivos = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .select(
            "id_motivo",
            "nome_motivo",
            "categoria",
            "criticidade",
            "ingestion_datetime"
        ).distinct()
        .orderBy("id_motivo")
)

In [0]:
df_motivos.count()

# Deduplicação

In [0]:
# Definição da janela agrupada pela chave primária e ordenada pelo timestamp decrescente
window_spec = Window.partitionBy("id_motivo").orderBy(F.col("ingestion_datetime").desc())

In [0]:
df_dedup = (
    df_motivos
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

# Tratamento

In [0]:
# Retira diferenças de caixa (minúsculas e Maiúsculas) da coluna de criticidade
df_tratamento = df_dedup.withColumn("criticidade", F.upper(F.trim(F.col("criticidade"))))

In [0]:
# Retira diferenças de caixa (minúsculas e Maiúsculas) da coluna de categoria
df_limpo = df_tratamento.withColumn("categoria", F.upper(F.trim(F.col("categoria"))))

In [0]:
df_limpo = df_limpo.withColumn(
    "categoria",
    F.when(F.col("nome_motivo") == "Contratação de cartão adicional", "CARTAO")
     .when(F.col("nome_motivo").isin("Transferência de agência", "Cancelamento de conta"), "CADASTRAL")
     .otherwise(F.col("categoria"))
)

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_motivos")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
save_df(df_limpo, silver_table_name)